# 01 — Data Audit and EDA

This notebook loads the untouched Seattle dataset, audits its quality, identifies leakage-prone variables, and performs initial exploratory analysis.

### Imports — why this cell is needed
Loads the libraries used for the raw-data audit and exploratory analysis.

In [ ]:
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

### Repository paths — why this cell is needed
Uses only paths relative to the repository structure required by the faculty.

In [ ]:
def find_repo_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from inside the cloned "
        "CSE437 repository, with data/ and notebooks/ folders present."
    )

ROOT = find_repo_root()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
FIGURES_DIR = ROOT / "figures"
MODELS_DIR = ROOT / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)

### Load the untouched raw dataset — why this cell is needed
Reads the original CSV from `data/raw/` without modifying it.

In [ ]:
csv_files = sorted(RAW_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(
        "No CSV was found in data/raw/. Put the original Seattle CSV there first."
    )

RAW_CSV = csv_files[0]
df_raw = pd.read_csv(RAW_CSV, low_memory=False)

print("Loaded:", RAW_CSV.name)
print("Raw shape:", df_raw.shape)
display(df_raw.head())

## Data audit

### Raw shape, duplicates, and missingness — why this cell is needed
Documents the condition of the original real-world dataset before any cleaning.

In [ ]:
print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])
print("Exact duplicate rows:", df_raw.duplicated().sum())

raw_audit = pd.DataFrame({
    "Data Type": df_raw.dtypes.astype(str),
    "Missing Count": df_raw.isna().sum(),
    "Missing %": (df_raw.isna().mean() * 100).round(2),
    "Unique Values": df_raw.nunique(dropna=True)
}).sort_values("Missing %", ascending=False)

display(raw_audit)

### Resolve important columns and leakage fields — why this cell is needed
Finds the real Seattle column names and identifies variables that must not be used as predictors.

In [ ]:
def normalize_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())

norm_map = {normalize_name(c): c for c in df_raw.columns}

def find_column(aliases, required=False):
    for alias in aliases:
        key = normalize_name(alias)
        if key in norm_map:
            return norm_map[key]
    if required:
        raise KeyError(
            f"Required column not found. Tried: {aliases}\n"
            f"Available columns: {list(df_raw.columns)}"
        )
    return None

TARGET_RAW = find_column(
    ["SiteEUI(kBtu/sf)", "SiteEUI", "siteeui_kbtu_sf"],
    required=True
)
GROUP_RAW = find_column(
    ["OSEBuildingID", "OSE Building ID", "osebuildingid"],
    required=True
)

COMPLIANCE_RAW = find_column(
    ["ComplianceStatus", "compliancestatus"]
)
COMPLIANCE_ISSUE_RAW = find_column(
    ["ComplianceIssue", "complianceissue"]
)
DEMOLISHED_RAW = find_column(
    ["Demolished", "demolished"]
)

FEATURE_ALIASES = {
    "DataYear": ["DataYear", "datayear"],
    "BuildingType": ["BuildingType", "buildingtype"],
    "EPAPropertyType": ["EPAPropertyType", "epapropertytype"],
    "LargestPropertyUseType": [
        "LargestPropertyUseType", "largestpropertyusetype"
    ],
    "SecondLargestPropertyUseType": [
        "SecondLargestPropertyUseType", "secondlargestpropertyusetype"
    ],
    "ThirdLargestPropertyUseType": [
        "ThirdLargestPropertyUseType", "thirdlargestpropertyusetype"
    ],
    "Neighborhood": ["Neighborhood", "neighborhood"],
    "CouncilDistrictCode": [
        "CouncilDistrictCode", "councildistrictcode"
    ],
    "ZipCode": ["ZipCode", "zipcode"],
    "YearBuilt": ["YearBuilt", "yearbuilt"],
    "NumberofFloors": ["NumberofFloors", "numberoffloors"],
    "NumberofBuildings": ["NumberofBuildings", "numberofbuildings"],
    "PropertyGFATotal": ["PropertyGFATotal", "propertygfatotal"],
    "PropertyGFABuilding": [
        "PropertyGFABuilding(s)", "PropertyGFABuilding",
        "propertygfabuildings"
    ],
    "PropertyGFAParking": [
        "PropertyGFAParking", "propertygfaparking"
    ],
    "LargestPropertyUseTypeGFA": [
        "LargestPropertyUseTypeGFA", "largestpropertyusetypegfa"
    ],


    "SecondLargestPropertyUseTypeGFA": [
        "SecondLargestPropertyUseTypeGFA",
        "secondlargestpropertyuse"
    ],
    "ThirdLargestPropertyUseTypeGFA": [
        "ThirdLargestPropertyUseTypeGFA",
        "thirdlargestpropertyusetypegfa"
    ],
    "Latitude": ["Latitude", "latitude"],
    "Longitude": ["Longitude", "longitude"]
}

resolved_features = {}
for clean_name, aliases in FEATURE_ALIASES.items():
    raw_name = find_column(aliases)
    if raw_name is not None:
        resolved_features[clean_name] = raw_name

LEAKAGE_PATTERNS = [
    "energystar",
    "siteeui",
    "sourceeui",
    "siteenergyuse",
    "electricity",
    "naturalgas",
    "steamuse",
    "otherfuel",
    "ghg",
    "emission"
]

leakage_columns = [
    c for c in df_raw.columns
    if any(pattern in normalize_name(c) for pattern in LEAKAGE_PATTERNS)
]

print("Target:", TARGET_RAW)
print("Building grouping ID:", GROUP_RAW)
print("Compliance status:", COMPLIANCE_RAW)
print("Compliance issue:", COMPLIANCE_ISSUE_RAW)
print("Demolished flag:", DEMOLISHED_RAW)

print("\nLeakage-related columns found and excluded from predictors:")
for col in leakage_columns:
    print(" -", col)

print("\nSafe predictor columns resolved:")
for clean_name, raw_name in resolved_features.items():
    print(f" - {clean_name} <- {raw_name}")


for raw_name in resolved_features.values():
    assert not any(
        pattern in normalize_name(raw_name)
        for pattern in LEAKAGE_PATTERNS
    ), f"Leakage predictor detected: {raw_name}"

### Inspect extreme raw SiteEUI records — why this cell is needed
Shows the extreme upper tail together with relevant quality/context fields before cleaning.

In [ ]:
audit_cols = [GROUP_RAW, TARGET_RAW]

for optional_col in [
    COMPLIANCE_RAW,
    COMPLIANCE_ISSUE_RAW,
    DEMOLISHED_RAW,
    resolved_features.get("BuildingType"),
    resolved_features.get("EPAPropertyType"),
    resolved_features.get("DataYear"),
    resolved_features.get("PropertyGFATotal")
]:
    if optional_col is not None and optional_col not in audit_cols:
        audit_cols.append(optional_col)

target_numeric = pd.to_numeric(df_raw[TARGET_RAW], errors="coerce")

target_outlier_audit = (
    df_raw.loc[target_numeric.notna(), audit_cols]
    .assign(_Target=target_numeric[target_numeric.notna()])
    .sort_values("_Target", ascending=False)
    .drop(columns="_Target")
    .head(20)
)

print("20 highest SiteEUI records in the raw dataset:")
display(target_outlier_audit)

print("\nRaw SiteEUI percentiles:")
display(
    target_numeric.dropna()
    .quantile([0.50, 0.90, 0.95, 0.99, 0.995, 0.999, 1.00])
    .to_frame("SiteEUI")
)

## Exploratory analysis

### Raw target and category overview — why this cell is needed
Provides a simple first look at the target distribution and the most common EPA property categories before preprocessing.

In [ ]:
target_raw_numeric = pd.to_numeric(df_raw[TARGET_RAW], errors="coerce")

print("Raw SiteEUI descriptive statistics:")
display(target_raw_numeric.describe().to_frame("SiteEUI"))

if resolved_features.get("EPAPropertyType") is not None:
    epa_col = resolved_features["EPAPropertyType"]
    print("\nMost common EPA property types:")
    display(df_raw[epa_col].value_counts(dropna=False).head(20).to_frame("Count"))

plt.figure(figsize=(8, 4))
plt.hist(target_raw_numeric.dropna(), bins=60)
plt.xlabel("SiteEUI")
plt.ylabel("Frequency")
plt.title("Raw SiteEUI Distribution")
plt.tight_layout()
plt.show()